In [1]:
import json
import random
import pandas as pd
from pathlib import Path

In [2]:
DATA_DIR = Path("data")

JSONL_PATH = DATA_DIR / "data.jsonl"
TRAIN_PATH = DATA_DIR / "train.txt"

In [3]:
code_map = {}

with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        code_map[item["idx"]] = item["func"]

In [4]:
train_pairs = []

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    for line in f:
        i1, i2, label = line.strip().split()
        train_pairs.append((str(i1), str(i2), int(label)))

In [11]:
idx1, idx2, label = random.choice(train_pairs)

code1 = code_map[idx1]
code2 = code_map[idx2]

print("Label:", label)
print("\n--- CODE 1 ---\n", code1[:300])
print("\n--- CODE 2 ---\n", code2[:300])

Label: 0

--- CODE 1 ---
     public final boolean login(String user, String pass) {
        if (user == null || pass == null) return false;
        connectionInfo.setData("com.tensegrity.palojava.pass#" + user, pass);
        try {
            MessageDigest md = MessageDigest.getInstance("MD5");
            md.update(pass.g

--- CODE 2 ---
     private final void reOrderFriendsListByOnlineStatus() {
        boolean flag = true;
        while (flag) {
            flag = false;
            for (int i = 0; i < friendsCount - 1; i++) if (friendsListOnlineStatus[i] < friendsListOnlineStatus[i + 1]) {
                int j = friendsListOnlin


In [12]:
from spectral_code.pipeline import Pipeline
from spectral_code.config import PipelineConfig

In [34]:
config = PipelineConfig(
    graph_type="ast",
    spectral_mode="laplacian",
    eigen_solver="sparse",
    k_eigen=20,
)

pipeline = Pipeline(config)

In [35]:
result1 = pipeline.run(code1, lang="java")
result2 = pipeline.run(code2, lang="java")

In [36]:
eig1 = result1["eigenvalues"]
eig2 = result2["eigenvalues"]

In [37]:
import numpy as np

# align lengths (important because eigsh returns k values)
k = min(len(eig1), len(eig2))

distance = np.linalg.norm(eig1[:k] - eig2[:k])

print("Spectral distance:", distance)
print("Ground truth label:", label)

Spectral distance: 9.28364356578905
Ground truth label: 0
